## Used Libraries

In [48]:
# Import libraries
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, kruskal
import statsmodels.api as sm
import statsmodels.formula.api as smf #Logistic Regression

## Load The Data

In [ ]:
# Load dataset
df = pd.read_excel("data/cleaned_data.xlsx")

# Show columns
print(df.columns)

# Show first rows
df.head()

Index(['Gender', 'What's Your Age?',
       'What is your current educational status?',
       'What is your field of study?',
       'Do you like the writing style of Artificial Intelligence (AI)?',
       'Which writing style gives you more trust?',
       'What are the main advantages of using AI in writing?',
       'Do you think AI can be relied upon in writing?',
       'Do you feel that AI writing is sometimes repetitive or similar?',
       'Do you think AI can convey emotions in writing?',
       'Rate AI writing when dealing with complex topics.',
       'How would you rate the creativity of AI-generated writing?',
       'Does AI writing need editing?',
       'What most distinguishes human writing?',
       'How would you rate the clarity and writing style of this',
       'In your opinion, who wrote this text?',
       'How would you rate the clarity and writing style of this   2',
       'In your opinion, who wrote this text?   2',
       'Which AI tools do you use for wr

,Gender,What's Your Age?,What is your current educational status?,What is your field of study?,Do you like the writing style of Artificial Intelligence (AI)?,Which writing style gives you more trust?,What are the main advantages of using AI in writing?,Do you think AI can be relied upon in writing?,Do you feel that AI writing is sometimes repetitive or similar?,Do you think AI can convey emotions in writing?,...,How would you rate the creativity of AI-generated writing?,Does AI writing need editing?,What most distinguishes human writing?,How would you rate the clarity and writing style of this,"In your opinion, who wrote this text?",How would you rate the clarity and writing style of this 2,"In your opinion, who wrote this text? 2",Which AI tools do you use for writing?,Which writing style gives you more trust? 2,response_id
0,Male,20 - 23,University,Data Science,I somewhat like it,Both equally,"Speed in completing tasks, Helps generate ideas",Only in informal writing,Often,"Yes, to a great extent",...,3,Often,"Emotional expression, Writer’s experience and ...",Very clear with a good style,Human,Very clear with a good style,Human,"ChatGPT, Anthropic Claude, Google Gemini",Human writing,2
1,Female,20 - 23,University,Data Science,I somewhat like it,Both equally,Saving effort,In both,Sometimes,To a limited extent,...,3,Often,Personal style,Very clear with a good style,Human,Very clear with a good style,Artificial Intelligence,ChatGPT,It depends on the topic,3
2,Female,20 - 23,University,Data Science,I somewhat like it,AI writing,"Language formulation/style, Helps generate ideas",Only in formal writing,Sometimes,To a limited extent,...,4,Often,"Emotional expression, Personal style",Moderately clear,Human,Very clear with a good style,Artificial Intelligence,"ChatGPT, Google Gemini",AI writing,4
3,Female,16 - 19,University,Data Science,I somewhat like it,It depends on the topic,Speed in completing tasks,In both,Often,To a limited extent,...,3,Often,Emotional expression,Moderately clear,Artificial Intelligence,Very clear with a good style,Artificial Intelligence,"Google Gemini, ChatGPT",Both equally,5
4,Female,16 - 19,University,Data Science,Neutral,Human writing,Speed in completing tasks,In both,Often,"Yes, to a moderate extent",...,4,Sometimes,Emotional expression,Moderately clear,Artificial Intelligence,Very clear with a good style,Not sure,"ChatGPT, Google Gemini",It depends on the topic,6


In [50]:
# Create new column:
# AI_Detection_Score
# In your opinion, who wrote this text?   2
# Q1 correct answer = AI
# Q2 correct answer = AI

# Replace with your real column names
q1 = "In your opinion, who wrote this text?"
q2 = "In your opinion, who wrote this text?   2"

# Convert answers to score
df["Q1_Correct"] = (df[q1] == "Artificial Intelligence").astype(int)
df["Q2_Correct"] = (df[q2] == "Artificial Intelligence").astype(int)

# Total score = 0 / 1 / 2
df["AI_Detection_Score"] = df["Q1_Correct"] + df["Q2_Correct"]

df[["Q1_Correct", "Q2_Correct", "AI_Detection_Score"]].head()

,Q1_Correct,Q2_Correct,AI_Detection_Score
0,0,0,0
1,0,1,1
2,0,1,1
3,1,1,2
4,1,0,1


## Age vs Ability to Detect AI

In [51]:
# Test if Age affects AI detection ability
# Using Kruskal-Wallis test

age_col = "What's Your Age?"

groups = []

for age_group in df[age_col].dropna().unique():
    scores = df[df[age_col] == age_group]["AI_Detection_Score"]
    groups.append(scores)

stat, p = kruskal(*groups)

print("Kruskal Statistic:", stat)
print("P-value:", p)

if p < 0.05:
    print("Significant relationship between Age and AI detection ability")
else:
    print("No significant relationship")

Kruskal Statistic: 3.852329103078735
P-value: 0.14570597589755027
No significant relationship


## Field of Study vs Ability to Detect AI

In [52]:
# Test if Field of Study affects detection ability

field_col = "What is your field of study?"

groups = []

for field in df[field_col].dropna().unique():
    scores = df[df[field_col] == field]["AI_Detection_Score"]
    groups.append(scores)

stat, p = kruskal(*groups)

print("Kruskal Statistic:", stat)
print("P-value:", p)

Kruskal Statistic: 7.707223775732712
P-value: 0.1731263788256199


## Age + Field of Study Together

In [53]:
# Logistic Regression
# Predict good detector using Age and Field

age_col = "What's Your Age?"
field_col = "What is your field of study?"

# Create binary target
df["Good_Detector"] = (df["AI_Detection_Score"] == 2).astype(int)

# Run logistic regression
formula = f"Good_Detector ~ C(Q(\"{age_col}\")) + C(Q(\"{field_col}\"))"

model = smf.logit(formula, data=df).fit()

print(model.summary())

         Current function value: 0.372245
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:          Good_Detector   No. Observations:                  256
Model:                          Logit   Df Residuals:                      248
Method:                           MLE   Df Model:                            7
Date:                Tue, 28 Apr 2026   Pseudo R-squ.:                 0.03138
Time:                        02:15:57   Log-Likelihood:                -95.295
converged:                      False   LL-Null:                       -98.381
Covariance Type:            nonrobust   LLR p-value:                    0.5197
                                                             coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------
Intercept                                               

c:\Users\thest\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## Field of Study vs AI Needs Editing

In [54]:
# Chi-Square Test
# Field of Study vs AI needs editing

edit_col = "Does AI writing need editing?"

table = pd.crosstab(df[field_col], df[edit_col])

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("P-value:", p)

Chi-square: 17.501247731879978
P-value: 0.6202266247515235


## Field of Study vs Creativity

In [55]:
# Kruskal Test
# Field of Study vs Creativity rating

creative_col = "How would you rate the creativity of AI-generated writing?"

groups = []

for field in df[field_col].dropna().unique():
    vals = df[df[field_col] == field][creative_col].dropna()
    groups.append(vals)

stat, p = kruskal(*groups)

print("Statistic:", stat)
print("P-value:", p)

Statistic: 3.2660670873680204
P-value: 0.6590406653438777


## Field of Study vs Complex Topics

In [56]:
# Kruskal Test
# Field of Study vs complex topics rating

complex_col = "Rate AI writing when dealing with complex topics."

groups = []

for field in df[field_col].dropna().unique():
    vals = df[df[field_col] == field][complex_col].dropna()
    groups.append(vals)

stat, p = kruskal(*groups)

print("Statistic:", stat)
print("P-value:", p)

Statistic: 6.813331957257112
P-value: 0.2348973228436682
